# 3. Sample model

A sample model is a structure in its buffer: a solvent, and maybe a salt. Here we build an asymmetric DOPC/POPC vesicle in TIP3 water with 0.15 mol/L NaCl.

In this notebook you will:

1. load the components from the library
2. set up the vesicle and its leaflets
3. set up the buffer
4. assemble the sample model, which runs GROMACS
5. look at the model and export it

Run the setup cell first. On Colab it installs GROMACS and Shapespyer, which takes about 3 minutes. Then run the helpers cell under it: it only shows and downloads results, and isn't part of the Shapespyer workflow.

In [ ]:
# @title Setup: on Colab, install GROMACS and Shapespyer (about 3 min) { display-mode: "form" }
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
CONFIG_FILE = Path("shapespyer.yaml")  # where Shapespyer keeps its files

if IN_COLAB:
    !apt-get -qq update > /dev/null && apt-get -qq install -y --no-install-recommends gromacs > /dev/null
    %pip install -q "shapespyer==0.3.0"

    # nglview draws molecules through Colab's widget manager
    from google.colab import output
    output.enable_custom_widget_manager()
    # and without its frame player: one-frame views don't use it, and its jslink opens Colab's widget panel
    import nglview
    nglview.NGLWidget._create_player = lambda self: None

    # the library, Temp and Projects go to /content, to browse them in the Files panel
    from shapes.project import Config
    config = Config(CONFIG_FILE)
    config.app_path = "/content/shapespyer"
    config.save()
elif not CONFIG_FILE.is_file():
    err_msg = f"No {CONFIG_FILE} in {Path.cwd()}; run the notebook in its own folder, next to shapespyer.yaml"
    raise FileNotFoundError(err_msg)

from shapes.project.files import AppFileManager

# the first call sets where the files go, for the whole session
config = AppFileManager(CONFIG_FILE).config
print("Shapespyer files:", config.app_path)
if config.gmx_problem is not None:
    err_msg = (
        f"This notebook needs GROMACS: {config.gmx_problem}. Install it, or set gmx_executable"
        f" in {CONFIG_FILE}. On Windows, the easy way is to run this notebook in WSL or on Colab."
    )
    raise RuntimeError(err_msg)
print("GROMACS:", config.gmx_version)

In [ ]:
# @title Helpers (not part of the Shapespyer workflow) { display-mode: "form" }
import shutil
from pathlib import Path

import mdtraj
import nglview


def show_structure(component):
    """Draw a component, as its .show() does; Shapespyer 0.3.0's .show() fails on Colab."""
    view = nglview.show_structure_file(str(component.structure_file_path))
    view.clear_representations()
    view.add_representation("ball+stick", selection=component.name, radius=0.25)
    return view


def _file_with_suffix(asset, suffix):
    found = [path for path in asset.paths if path.suffix == suffix]
    if not found:
        raise FileNotFoundError(f"{asset.name} holds no {suffix} file yet")
    return found[0]


def _molecules(topology, resname):
    """Atom indices of the residues named resname (a name or a list of names), or of all but the water."""
    # mdtraj renames TIP3 to HOH on loading, so select it with the "water" keyword
    if resname is None:
        return topology.select("not water")
    names = [resname] if isinstance(resname, str) else list(resname)
    atoms = topology.select(" or ".join(f"resname {name}" for name in names))
    if len(atoms) == 0:
        found = sorted({residue.name for residue in topology.residues})
        raise ValueError(f"No residues named {', '.join(names)}; the file has {', '.join(found)}")
    return atoms


def show_without_water(asset, resname=None):
    """Draw the .gro an asset holds, such as a sample model or a stage, without the water.

    Give resname, a residue name or a list of them, to draw only those molecules.
    """
    frame = mdtraj.load(_file_with_suffix(asset, ".gro"))
    shown = frame.atom_slice(_molecules(frame.top, resname))
    print(f"{frame.n_atoms} atoms, box {frame.unitcell_lengths[0]} nm; {shown.n_atoms} shown")
    return nglview.show_mdtraj(shown)


def list_files(folder):
    """Print the files under a folder, relative to it."""
    for path in sorted(folder.rglob("*")):
        if path.is_file():
            print(path.relative_to(folder))


def download_as_zip(folder):
    """Zip a folder into <folder>.zip next to it; on Colab, also download the zip."""
    archive = Path(shutil.make_archive(str(folder), "zip", folder))
    if IN_COLAB:
        from google.colab import files
        files.download(str(archive))
    return archive

In [ ]:
import shutil

from shapes.basics.utils import LogConfiguration
from shapes.project import Ion, Library, Lipid, SampleModel, Vesicle
from shapes.project.species import Buffer, Salt, Solvent

LogConfiguration("WARNING")

library = Library()
library.clean_temp()
# the export and its zip go to a work folder, which the clean up deletes
WORK = config.app_path / "work"

### Load the components

In [ ]:
dopc = Lipid.load("DOPC")
popc = Lipid.load("POPC")
show_structure(dopc)

In [ ]:
show_structure(popc)

### Set up the vesicle

A component added to the structure joins every layer.

In [ ]:
vesicle = Vesicle(name="DemoVesicle")
vesicle.components.append(dopc)
vesicle.components.append(popc)
vesicle.fill = "FIBO"

print(vesicle)

### One lamella with different leaflets

- inner leaflet: POPC and DOPC in the ratio 2 : 3
- outer leaflet: DOPC only (POPC is marked absent)

In [ ]:
lamella = vesicle.lamellae[0]
lamella.symmetric = False
lamella.rmin = 5
lamella.shell_thickness = 5

lamella.inner_leaflet_composition.set_mol_ratio("POPC", 2)
lamella.inner_leaflet_composition.set_mol_ratio("DOPC", 3)
lamella.inner_leaflet_dmin = 1.1

lamella.outer_leaflet_composition.set_presence("POPC", False)
lamella.outer_leaflet_composition.set_mol_ratio("DOPC", 100)
lamella.outer_leaflet_dmin = 2

print(vesicle.lamellae)

### Set up the buffer

TIP3 water with 0.15 mol/L NaCl. DOPC and POPC are neutral, so no counter ions are needed.

In [ ]:
buffer = Buffer(name="DemoBuffer", solvent=Solvent.load("TIP3"))
buffer.components.append(
    Salt(name="NaCl", cation=Ion.load("NAT"), anion=Ion.load("CLA"), concentration=0.15)
)

model = SampleModel(vesicle, buffer=buffer, name="DemoModel")
print(model)

### Assemble the sample model

`.assemble()` builds the vesicle, puts in the salt ions with `gmx insert-molecules` and fills the box with water with `gmx solvate`. Then the model holds its `.gro`, its `.top` and `assembly.log`.

If you change any setting of the model or the vesicle, call `.assemble()` again.

In [ ]:
model.assemble()
model.paths

### Look at the model

We hide the water, so the vesicle and the ions are easy to see.

In [ ]:
show_without_water(model)

### Export and download

`export_to()` copies the files of the model, and of every asset in it, into a folder you choose, here `DemoModel/` in the work folder. That keeps the model to reuse: `SampleModel.import_from()` loads it back from the folder. The vesicle is assembled on its own too, so the export also holds the vesicle's own `.gro`. The cell after it zips the export; on Colab it also downloads the zip.

In [ ]:
vesicle.assemble()
model.export_to(WORK / "DemoModel", overwrite=True)
list_files(WORK / "DemoModel")

In [ ]:
download_as_zip(WORK / "DemoModel")

### Clean up (optional)

This deletes the work folder, with the export and its zip, and Temp, where the vesicle and the model were assembled. `force=True` is needed for Temp, because the assets of this session still hold their files there.

In [ ]:
shutil.rmtree(WORK, ignore_errors=True)
library.clean_temp(force=True)
print(WORK.exists())